# 🎭 Day 04a: Polymorphism — One Interface, Many Forms

---

## 🎯 What You'll Master Today
- Compile-time (method overloading) vs runtime (method overriding) polymorphism
- Duck typing — Python's approach to polymorphism
- Protocol classes — structural subtyping
- Operator overloading via dunder methods
- Write polymorphic code confidently in interviews

---

## 🎭 The Analogy

Think of a **universal remote**:

- Same button: "Power" — but it turns on a TV, AC, or speaker depending on what it's pointed at
- Same interface (press power), different behavior (each device responds differently)
- That's **polymorphism**: one interface, many implementations

In Python: "If it walks like a duck and quacks like a duck, it's a duck."  
→ We don't check the type, we check the **behavior**.

---

## 📐 Types of Polymorphism

```
╔═══════════════════════════════════════════════════════════════════╗
║  POLYMORPHISM                                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. COMPILE-TIME (Static)          2. RUNTIME (Dynamic)         ║
║     Method Overloading                Method Overriding          ║
║     • Same name, different params     • Same name, same params  ║
║     • Resolved at compile time        • Resolved at runtime     ║
║     • Python: NOT natively supported  • Python: ✅ supported    ║
║       (use default args / *args)       (inheritance + override) ║
║                                                                   ║
║  3. DUCK TYPING (Python's way)     4. OPERATOR OVERLOADING     ║
║     • Don't check type, check         • +, -, ==, <, [], etc.  ║
║       behavior                        • Via dunder methods      ║
║     • "If it quacks like a duck..."   • __add__, __eq__, etc.  ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Runtime Polymorphism — Method Overriding
# ============================================
from abc import ABC, abstractmethod

class Notification(ABC):
    """Base class — defines the interface."""
    
    @abstractmethod
    def send(self, message: str) -> str:
        pass


class EmailNotification(Notification):
    def send(self, message: str) -> str:
        return f"📧 Email: {message}"


class SMSNotification(Notification):
    def send(self, message: str) -> str:
        return f"📱 SMS: {message}"


class PushNotification(Notification):
    def send(self, message: str) -> str:
        return f"🔔 Push: {message}"


# Polymorphism in action — same method, different behavior
def notify_user(notifier: Notification, msg: str):
    """Works with ANY notification type — doesn't know or care which."""
    print(notifier.send(msg))


# Same function, different objects → different behavior
notifiers = [
    EmailNotification(),
    SMSNotification(),
    PushNotification()
]

for n in notifiers:
    notify_user(n, "Your order has been shipped!")

In [ ]:
# ============================================
# 2. Duck Typing — Python's Core Polymorphism
# ============================================
# "If it walks like a duck and quacks like a duck..."
# No inheritance needed! Just implement the right methods.

class Duck:
    def quack(self):
        return "Quack! 🦆"
    def walk(self):
        return "Waddle waddle"

class Person:
    def quack(self):
        return "I'm quacking like a duck! 🧑"
    def walk(self):
        return "Walking normally"

class Robot:
    def quack(self):
        return "QUACK QUACK 🤖"
    def walk(self):
        return "Mechanical walking"


def make_it_quack(thing):  # No type annotation needed!
    """Doesn't care WHAT thing is, just that it can quack."""
    print(thing.quack())


# All three work — no shared base class, no inheritance!
for obj in [Duck(), Person(), Robot()]:
    make_it_quack(obj)

# This is duck typing: we check capabilities, not types
# Built-in examples: len(), iter(), str() — they don't check type,
# they check if __len__, __iter__, __str__ exist

In [ ]:
# ============================================
# 3. Protocol — Structural Subtyping (Python 3.8+)
# ============================================
from typing import Protocol, runtime_checkable

@runtime_checkable
class Drawable(Protocol):
    """Any class with a draw() method is considered Drawable.
    No inheritance needed — structural subtyping."""
    
    def draw(self) -> str:
        ...


class Circle:
    def draw(self) -> str:
        return "Drawing ⭕"

class Square:
    def draw(self) -> str:
        return "Drawing ⬜"

class NotDrawable:
    def paint(self) -> str:
        return "Painting"


# Protocol check — isinstance works with @runtime_checkable!
print(f"Circle is Drawable? {isinstance(Circle(), Drawable)}")       # True
print(f"Square is Drawable? {isinstance(Square(), Drawable)}")       # True
print(f"NotDrawable is Drawable? {isinstance(NotDrawable(), Drawable)}")  # False


def render(shape: Drawable):
    """Type checker knows shape has draw(). No inheritance required."""
    print(shape.draw())

render(Circle())
render(Square())

In [ ]:
# ============================================
# 4. Method Overloading — Python Style
# ============================================
# Python doesn't support traditional overloading.
# Instead: default args, *args, singledispatch

from functools import singledispatch

# Approach 1: Default arguments
class Calculator:
    def add(self, a, b=0, c=0):
        return a + b + c

calc = Calculator()
print(f"add(5):      {calc.add(5)}")        # 5
print(f"add(5, 3):   {calc.add(5, 3)}")     # 8
print(f"add(5, 3, 2): {calc.add(5, 3, 2)}")  # 10

# Approach 2: @singledispatch — dispatch based on first arg type
@singledispatch
def process(data):
    """Default handler."""
    print(f"Unknown type: {type(data).__name__}")

@process.register(str)
def _(data):
    print(f"Processing string: '{data}'")

@process.register(int)
def _(data):
    print(f"Processing integer: {data}")

@process.register(list)
def _(data):
    print(f"Processing list of {len(data)} items")


process("hello")    # string handler
process(42)          # int handler
process([1, 2, 3])   # list handler
process(3.14)        # default handler

In [ ]:
# ============================================
# 5. Operator Overloading — Make Objects Pythonic
# ============================================

class Money:
    """Interview example: implement a Money class with operators."""
    
    def __init__(self, amount: float, currency: str = "INR"):
        self.amount = amount
        self.currency = currency
    
    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError(f"Can't add {self.currency} and {other.currency}")
        return Money(self.amount + other.amount, self.currency)
    
    def __sub__(self, other):
        if self.currency != other.currency:
            raise ValueError(f"Can't subtract different currencies")
        return Money(self.amount - other.amount, self.currency)
    
    def __mul__(self, factor: float):
        return Money(self.amount * factor, self.currency)
    
    def __eq__(self, other):
        return self.amount == other.amount and self.currency == other.currency
    
    def __lt__(self, other):
        if self.currency != other.currency:
            raise ValueError("Can't compare different currencies")
        return self.amount < other.amount
    
    def __repr__(self):
        return f"Money({self.amount}, '{self.currency}')"
    
    def __str__(self):
        symbol = {"INR": "₹", "USD": "$", "EUR": "€"}.get(self.currency, self.currency)
        return f"{symbol}{self.amount:,.2f}"


salary = Money(150000, "INR")
bonus = Money(25000, "INR")
tax = Money(20000, "INR")

total = salary + bonus - tax
doubled = salary * 2

print(f"Salary: {salary}")
print(f"Total:  {total}")
print(f"Doubled: {doubled}")
print(f"Salary > Bonus? {salary > bonus}")
print(f"Salary == Money(150000, 'INR')? {salary == Money(150000, 'INR')}")

In [ ]:
# ============================================
# 6. Polymorphism in Built-in Python
# ============================================
# Python uses polymorphism EVERYWHERE

# len() works on anything with __len__
print(f"len('hello')  = {len('hello')}")     # str.__len__
print(f"len([1,2,3])  = {len([1,2,3])}")     # list.__len__
print(f"len({{'a':1}}) = {len({'a':1})}")      # dict.__len__

# + operator works differently based on type
print(f"\n3 + 4      = {3 + 4}")              # int.__add__
print(f"'ab' + 'cd' = {'ab' + 'cd'}")         # str.__add__
print(f"[1] + [2]   = {[1] + [2]}")           # list.__add__

# for loop works on anything with __iter__
print("\nfor loop polymorphism:")
for x in [1, 2, 3]:      pass  # list.__iter__
for x in "abc":           pass  # str.__iter__
for x in {1, 2, 3}:      pass  # set.__iter__
for x in {"a": 1}:       pass  # dict.__iter__
print("  All work with the same 'for' syntax!")

## 🗺️ Polymorphism Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  POLYMORPHISM — QUICK REFERENCE                                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  RUNTIME: Method overriding via inheritance                      ║
║  DUCK TYPING: Check behavior, not type (Pythonic way)           ║
║  PROTOCOL: Structural subtyping (no inheritance needed)          ║
║  OPERATOR: Dunder methods (__add__, __eq__, etc.)               ║
║                                                                   ║
║  Python has NO true method overloading.                          ║
║  Use: default args, *args, @singledispatch                      ║
║                                                                   ║
║  Built-in polymorphism: len(), +, for, in — all dispatch        ║
║  based on the type's dunder methods.                             ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions — Practice These

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is polymorphism? | Same interface, different behavior. One method name, many implementations |
| 2 | Compile-time vs runtime polymorphism? | Compile-time = overloading (Python doesn't have). Runtime = overriding (Python uses) |
| 3 | What is duck typing? | If an object has the right methods, it's the right type. No inheritance check needed |
| 4 | Protocol vs ABC? | ABC = nominal (must inherit). Protocol = structural (just implement the methods) |
| 5 | Does Python support method overloading? | Not natively. Use default args, *args, or @singledispatch |
| 6 | What is operator overloading? | Defining behavior for +, -, ==, etc. via dunder methods |
| 7 | How does len() work on different types? | Polymorphism — calls __len__ on whatever object you pass |
| 8 | What is @singledispatch? | Function overloading by type of first argument — standard library solution |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Polymorphism = same interface, different behavior     │
## │  • Python uses duck typing over strict type checking     │
## │  • Method overriding = runtime polymorphism              │
## │  • Protocol = structural subtyping (no inheritance)      │
## │  • Operator overloading via dunder methods               │
## │  • Python built-ins are polymorphic (len, +, for, in)   │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **SOLID Principles — SRP & OCP** — writing maintainable code

# 🧠 Polymorphism — The Power Move

**"Many forms"** — Same interface, different behavior.

Think of the `+` operator:
- `2 + 3` → `5` (addition)
- `"hi" + " there"` → `"hi there"` (concatenation)
- `[1,2] + [3,4]` → `[1,2,3,4]` (list merge)

**Same symbol, totally different behavior** — that's polymorphism.

```
                    POLYMORPHISM
                    "Many Forms"
                         |
          +--------------+--------------+
          |              |              |
    Method           Operator       Duck
    Overriding       Overloading    Typing
    (runtime)        (__add__)      (quack!)
```

👉 **Why care?** Write ONE function that works with MANY types. Less code, more power.

---
## 🎯 1. Method Overriding (Runtime Polymorphism)

Parent defines a method → Child **replaces** it with its own version.

```
  Shape.area()          <-- parent blueprint
       |
  +----+----+
  |         |
Circle    Rectangle
.area()   .area()        <-- each child has OWN version
πr²       l × w
```

👉 The **caller doesn't care** which shape — just calls `.area()`

In [ ]:
import math

class Shape:
    def area(self):
        raise NotImplementedError("Subclass must implement")
    
    def __str__(self):
        return f"{self.__class__.__name__}: area={self.area():.2f}"

class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius
    
    def area(self):  # OVERRIDES parent
        return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height
    
    def area(self):  # OVERRIDES parent
        return self.width * self.height

class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height
    
    def area(self):  # OVERRIDES parent
        return 0.5 * self.base * self.height

In [ ]:
# 👉 THE POLYMORPHIC FUNCTION — doesn't care which shape!
def print_total_area(shapes: list[Shape]):
    total = 0
    for s in shapes:
        print(f"  {s}")           # calls each shape's __str__ → area()
        total += s.area()         # calls the RIGHT area() automatically
    print(f"  Total area: {total:.2f}")

shapes = [Circle(5), Rectangle(4, 6), Triangle(3, 8)]
print_total_area(shapes)

---
## 🎯 2. Operator Overloading

Make your objects work with `+`, `*`, `==`, `<`, `[]`, `len()` etc.

```
  Operator    │  Dunder Method
  ────────────┼────────────────
  a + b       │  a.__add__(b)
  a * b       │  a.__mul__(b)
  a == b      │  a.__eq__(b)
  a < b       │  a.__lt__(b)
  a[i]        │  a.__getitem__(i)
  len(a)      │  a.__len__()
  str(a)      │  a.__str__()
  repr(a)     │  a.__repr__()
```

👉 You're teaching Python: "hey, when someone does `v1 + v2`, call THIS method"

In [ ]:
class Vector:
    def __init__(self, *components):
        self.components = list(components)
    
    # --- Operator Overloading ---
    def __add__(self, other):       # v1 + v2
        return Vector(*(a + b for a, b in zip(self.components, other.components)))
    
    def __mul__(self, scalar):      # v * 3
        return Vector(*(x * scalar for x in self.components))
    
    def __rmul__(self, scalar):     # 3 * v  (reverse)
        return self.__mul__(scalar)
    
    def __eq__(self, other):        # v1 == v2
        return self.components == other.components
    
    def __lt__(self, other):        # v1 < v2 (by magnitude)
        return self.magnitude() < other.magnitude()
    
    def __getitem__(self, index):   # v[0], v[1]
        return self.components[index]
    
    def __len__(self):              # len(v)
        return len(self.components)
    
    def magnitude(self):
        return sum(x**2 for x in self.components) ** 0.5
    
    def __repr__(self):
        return f"Vector{tuple(self.components)}"

In [ ]:
v1 = Vector(1, 2, 3)
v2 = Vector(4, 5, 6)

print(f"v1 + v2  = {v1 + v2}")      # __add__
print(f"v1 * 3   = {v1 * 3}")       # __mul__
print(f"3 * v1   = {3 * v1}")       # __rmul__
print(f"v1 == v2 = {v1 == v2}")     # __eq__
print(f"v1 < v2  = {v1 < v2}")      # __lt__
print(f"v1[0]    = {v1[0]}")        # __getitem__
print(f"len(v1)  = {len(v1)}")      # __len__

---
## 🎯 3. Duck Typing — The Pythonic Way

> "If it walks like a duck and quacks like a duck, it's a duck."

```
  Dog.speak()  → "Woof!"     ┐
  Cat.speak()  → "Meow!"     ├── No inheritance needed!
  Duck.speak() → "Quack!"    ┘   Just have .speak()
```

👉 Python doesn't check TYPE — it checks **CAPABILITY**. If the object has the method, it works.

In [ ]:
# NO shared parent class — just same method name
class Dog:
    def speak(self): return "Woof!"

class Cat:
    def speak(self): return "Meow!"

class Robot:
    def speak(self): return "Beep boop!"

# This function doesn't care about TYPE — just that .speak() exists
def make_it_speak(thing):
    print(f"{thing.__class__.__name__} says: {thing.speak()}")

for obj in [Dog(), Cat(), Robot()]:
    make_it_speak(obj)

In [ ]:
# 👉 Real-world duck typing: anything with __len__ works with len()
class Playlist:
    def __init__(self, songs):
        self.songs = songs
    def __len__(self):
        return len(self.songs)
    def __getitem__(self, i):
        return self.songs[i]

p = Playlist(["Song A", "Song B", "Song C"])
print(f"Playlist length: {len(p)}")   # works! duck typing
print(f"First song: {p[0]}")          # works! duck typing

# Can even iterate — because __getitem__ is enough!
for song in p:
    print(f"  ♪ {song}")

---
## 🎯 4. `@functools.singledispatch` — Function Overloading in Python

Python doesn't natively support function overloading (same name, diff params).

But `singledispatch` lets you dispatch based on the **type of the first argument**.

```
  process(int)   → "Processing number"
  process(str)   → "Processing text"
  process(list)  → "Processing list"
```

In [ ]:
from functools import singledispatch

@singledispatch
def process(data):
    """Default handler — unknown type"""
    print(f"  Don't know how to process {type(data).__name__}")

@process.register(int)
def _(data):
    print(f"  Crunching number: {data * 2}")

@process.register(str)
def _(data):
    print(f"  Processing text: '{data.upper()}'")

@process.register(list)
def _(data):
    print(f"  Processing list of {len(data)} items: {data}")

@process.register(dict)
def _(data):
    print(f"  Processing dict with keys: {list(data.keys())}")

# Same function name — different behavior per type!
process(42)
process("hello")
process([1, 2, 3])
process({"a": 1, "b": 2})
process(3.14)  # falls to default

---
## 🎯 5. Abstract Methods as Polymorphism Contract

ABCs **force** subclasses to implement methods → guaranteed polymorphism.

```
  PaymentProcessor (ABC)
  ├── process_payment()  ← MUST implement
  └── refund()           ← MUST implement
       |
  +----+--------+--------+
  |             |        |
  CreditCard  PayPal  Crypto
```

In [ ]:
from abc import ABC, abstractmethod

class PaymentProcessor(ABC):
    @abstractmethod
    def process_payment(self, amount: float) -> str:
        pass
    
    @abstractmethod
    def refund(self, amount: float) -> str:
        pass

class CreditCard(PaymentProcessor):
    def process_payment(self, amount):
        return f"💳 Charged ${amount:.2f} to credit card"
    def refund(self, amount):
        return f"💳 Refunded ${amount:.2f} to credit card"

class PayPal(PaymentProcessor):
    def process_payment(self, amount):
        return f"🅿️ Sent ${amount:.2f} via PayPal"
    def refund(self, amount):
        return f"🅿️ PayPal refund of ${amount:.2f}"

class Crypto(PaymentProcessor):
    def process_payment(self, amount):
        return f"₿ Transferred ${amount:.2f} in crypto"
    def refund(self, amount):
        return f"₿ Crypto refund of ${amount:.2f}"

In [ ]:
# 👉 Polymorphic checkout — doesn't care which payment method
def checkout(processor: PaymentProcessor, amount: float):
    print(processor.process_payment(amount))

processors = [CreditCard(), PayPal(), Crypto()]
for p in processors:
    checkout(p, 99.99)

print("\n--- Refunds ---")
for p in processors:
    print(p.refund(25.00))

# This would FAIL — can't instantiate abstract class
try:
    bad = PaymentProcessor()
except TypeError as e:
    print(f"\n❌ {e}")

---
## 🎯 6. Compile-time vs Runtime Polymorphism

This comes up in interviews (Java/C++ context). Python only has runtime.

```
  ┌─────────────────────┬────────────────────────┐
  │   COMPILE-TIME      │     RUNTIME            │
  │   (Static)          │     (Dynamic)          │
  ├─────────────────────┼────────────────────────┤
  │ Method Overloading  │ Method Overriding      │
  │ (same name, diff    │ (child replaces parent │
  │  params)            │  method)               │
  ├─────────────────────┼────────────────────────┤
  │ Resolved at         │ Resolved at            │
  │ COMPILE time        │ RUNTIME                │
  ├─────────────────────┼────────────────────────┤
  │ Java: int add(int)  │ Java: @Override        │
  │       int add(int,  │ Python: just redefine  │
  │            int)     │   the method           │
  ├─────────────────────┼────────────────────────┤
  │ ❌ Python: NO       │ ✅ Python: YES         │
  │ (use default args   │ (this is THE way)      │
  │  or singledispatch) │                        │
  └─────────────────────┴────────────────────────┘
```

👉 **Interview answer:** Python is dynamically typed, so all polymorphism is runtime. For compile-time-like behavior, use `@singledispatch` or default arguments.

In [ ]:
# Python's "overloading" workaround — default args
class Calculator:
    def add(self, a, b=0, c=0):  # "overloaded" via defaults
        return a + b + c

calc = Calculator()
print(calc.add(5))         # 5
print(calc.add(5, 3))      # 8
print(calc.add(5, 3, 2))   # 10

# Or use *args for unlimited
class FlexCalc:
    def add(self, *args):
        return sum(args)

fc = FlexCalc()
print(fc.add(1, 2, 3, 4, 5))  # 15

---
## 🗺️ Polymorphism Map

```
┌──────────────────────────────────────────────────────────┐
│                    POLYMORPHISM                          │
│                   "Many Forms"                           │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  ┌──────────────────┐  ┌──────────────────┐             │
│  │ METHOD OVERRIDING │  │ OPERATOR         │             │
│  │ (Runtime)         │  │ OVERLOADING      │             │
│  │                   │  │                  │             │
│  │ • Child replaces  │  │ • __add__ → +    │             │
│  │   parent method   │  │ • __eq__  → ==   │             │
│  │ • super() calls   │  │ • __lt__  → <    │             │
│  │   parent version  │  │ • __len__ → len()│             │
│  └──────────────────┘  └──────────────────┘             │
│                                                          │
│  ┌──────────────────┐  ┌──────────────────┐             │
│  │ DUCK TYPING      │  │ SINGLEDISPATCH   │             │
│  │ (Pythonic!)       │  │ (Func Overload)  │             │
│  │                   │  │                  │             │
│  │ • No inheritance  │  │ • @singledispatch│             │
│  │   needed          │  │ • Dispatch on    │             │
│  │ • Just implement  │  │   first arg type │             │
│  │   the method      │  │ • .register()    │             │
│  └──────────────────┘  └──────────────────┘             │
│                                                          │
│  ┌──────────────────┐  ┌──────────────────┐             │
│  │ ABC CONTRACT     │  │ COMPILE vs       │             │
│  │                   │  │ RUNTIME          │             │
│  │ • @abstractmethod │  │                  │             │
│  │ • Forces impl    │  │ • Python = ALL   │             │
│  │ • Can't skip     │  │   runtime        │             │
│  │ • Design contract│  │ • Java/C++ = both│             │
│  └──────────────────┘  └──────────────────┘             │
│                                                          │
└──────────────────────────────────────────────────────────┘
```

---
## 💡 Interview Quick-Fire

**Q1: What is polymorphism?**
> Same interface, different behavior. Objects of different types respond to the same method call differently.

**Q2: Method overloading vs overriding?**
> **Overloading** = same name, different params (compile-time, Java/C++). **Overriding** = child replaces parent method (runtime). Python only has overriding natively.

**Q3: What is duck typing?**
> Python checks capability, not type. If an object has the required method, it works — no inheritance needed. "If it quacks like a duck..."

**Q4: How does Python handle function overloading?**
> No native support. Use: (1) default arguments, (2) `*args/**kwargs`, (3) `@functools.singledispatch`, (4) `@typing.overload` for type hints.

**Q5: What's `@singledispatch`?**
> A decorator that dispatches to different function implementations based on the type of the first argument. Python's way of doing function overloading.

**Q6: How do ABCs enforce polymorphism?**
> `@abstractmethod` forces subclasses to implement methods. You can't instantiate the ABC — it's a contract guaranteeing the interface exists.

**Q7: What dunder methods enable operator overloading?**
> `__add__` (+), `__sub__` (-), `__mul__` (*), `__eq__` (==), `__lt__` (<), `__getitem__` ([]), `__len__` (len()), `__str__` (str()), `__repr__` (repr()).

**Q8: Compile-time vs runtime polymorphism in Python?**
> Python is dynamically typed — ALL polymorphism is runtime. Compile-time polymorphism (method overloading) exists in Java/C++ where the compiler picks the right method based on argument types.